## Importing Functions

In [ ]:
# Importing general libraries
import os
import numpy as np
import gspread
import pandas as pd
import emoji
import colorsys
import datetime
import yaml

# Importing plotting libraries
import matplotlib.pyplot as plt
from bokeh.plotting import figure, output_file, show, ColumnDataSource
from bokeh.io import output_notebook
from bokeh.palettes import magma

## Libraries & service account

In [ ]:
# Importing notebook output
output_notebook()

In [ ]:
# Function for creating dictionary of sheet data
def sheet_to_matrix(sheet, remove_nan = False):

    # Getting sheet
    current_sheet = spreadsheet.worksheet(sheet)

    # Creating matrix from the values
    current_matrix = current_sheet.get_all_values()
    
    # If remove_nan is True, only adding lists to new list which don't have '' or nan values
    if remove_nan == True:
        
        # Creating new empty list for non nan rows
        current_matrix_no_nan = []
    
        # Iterating through rows, removing nan values
        for i in range(0,len(current_matrix)):
            
            # Getting current row
            curr_row = current_matrix[i]
            
            # Removing empty values from current row
            while '' in curr_row:
                curr_row.remove('')              
                
            # If it's not a name column, append
            if len(curr_row) >= 8:
                current_matrix_no_nan.append(curr_row)
                
        # Returning sheet matrix without nan
        return current_matrix_no_nan 
    
    # Creating matrices for breakfast, lunch, dinner
    workout_matrix = []
    breakfast_matrix = []
    lunch_matrix = []
    dinner_matrix = []
    
    # Iterating through data, organising them into lists according to the meal type
    for i in range(0,len(current_matrix)):
        
        # Saving list as temp var
        current_row = current_matrix[i]
        
        # Removing empty values
        current_row = [x for x in current_row if x != '']
        
        # If current row is not empty, append
        if len(current_row) > 2:
            
            # If general
            if current_row[1] == 'general':
                del current_row[1]
                workout_matrix.append(current_row)
        
            # If breakfast
            elif current_row[1] == 'breakfast':

                # Deleting meal type indicator, adding to return list
                del current_row[1]
                breakfast_matrix.append(current_row)

            # If lunch
            elif current_row[1] == 'lunch':

                # Deleting meal type indicator, adding to return list
                del current_row[1]
                lunch_matrix.append(current_row)

            # If dinner
            elif current_row[1] == 'dinner':

                # Deleting meal type indicator, adding to return list
                del current_row[1]
                dinner_matrix.append(current_row)
    
    # Returning with the sheet matrix
    return workout_matrix, breakfast_matrix, lunch_matrix, dinner_matrix

In [ ]:
# Function for transforming sheet matrix to dictionary
def sheetmatrix_to_dict(matrix):

    # Creating dict from all items
    food_eaten_dict = {}

    # Adding items to breakfast dict
    for i in range(0,len(matrix)):

        # Getting row and date
        current_row = matrix[i]
        current_date = current_row[0]

        # Removing nan from list
        current_row = [x for x in current_row if x != '']

        # Creating dict list
        food_eaten_dict[current_date] = []

        # Iterating through all food items
        for j in range(1,len(current_row)):

            # Getting items
            if '-' in current_row[j]:
                current_item_pack = current_row[j].split(' - ')
                current_item = current_item_pack[0].lower()
                current_quant = current_item_pack[1]
            else:
                current_item = current_row[j]
                current_quant = "1"

            # Adding to dict
            food_eaten_dict[current_date].append({'item' : current_item, 'quantity' : current_quant})
                        
    # Returning the dictionary
    return(food_eaten_dict)

In [ ]:
# Function for transforming general info to dictionary
def general_info_to_dict(general_matrix, properties_list, cardio_list = [], calories_burnt_list = []):
    
    # List for ordered dates
    ordered_dates_list = []
    
    # Creating general dict
    general_dict = {}
    
    # Iterating through all rows, appending to dict
    for i in range(0,len(general_matrix)):
        
        # Creating key
        key = general_matrix[i][0].lower()

        # Appending to dates list
        ordered_dates_list.append(key)

        # Creating temp dict
        dict_temp = {}
        for cardio in cardio_list:
            dict_temp[cardio] = '-'

        # Iterating through columns
        for j in range(0,len(general_matrix[i])-1):   

            # Creating var for current column
            col = general_matrix[i][j+1]

            # If the column is not cardio, simply append:
            if col == '-' or '-' not in col:
                dict_temp[properties_list[j]] = col    

            # Else iterating through and extracting time and calories
            else:
                # Iterating through all cardio
                for k in range(0,len(cardio_list)):

                    # If cardio is present, append to cardio kcal
                    if cardio_list[k] in col:

                        # Removing name
                        cardio_time = col.split(' - ')[1]
                        cardio_min = float(cardio_time.split(' ')[0])

                        # Calculating calories
                        cardio_kcal = int(round(cardio_min*calories_burnt_list[k]))

                        # Getting different values from cardio, adding to sub-dict
                        dict_temp2 = {'time' : cardio_time, 'kcal' : cardio_kcal}

                        # Adding to column dictionary
                        dict_temp[cardio_list[k]] = dict_temp2

        # Adding temp dict to general matrix dict
        general_dict[key] = dict_temp
        
    # Returning with the food dictionary
    return general_dict, ordered_dates_list

In [ ]:
def getting_total_macros_cals(meal, dates):
    
    # Creating return matrix
    return_dict = {}
    
    # Iterating through all date entries
    for i in range(0,len(dates)):
        
        # Creating variable for calories, carbs, fat, protein
        total_calories = 0
        total_carbs = 0
        total_fat = 0
        total_protein = 0
        total_sugar = 0
        total_fiber = 0
        
        # If the meal is not empty:
        if dates[i] in meal:
            
            # Getting current meal row
            current_meal_row = meal[dates[i]]
        
            # Iterating through breakfast row
            for j in range(0,len(current_meal_row)):

                # Getting food items and quantity
                item = current_meal_row[j]['item'].strip()
                quantity = float(current_meal_row[j]['quantity'].replace('g', '').strip())

                # Error message
                if item not in all_food_dict:
                    print('**************************************')
                    print('Please add', item, 'to all food sheet.')
                    print('**************************************')

                # Calculating ratio of eaten food and dict saved quantity
                norm_quantity = float(all_food_dict[item]['quantity'].replace('g', '').replace('piece', '').strip())
                ratio = quantity / norm_quantity

                # Getting calories of item eaten
                total_calories += ratio * float(all_food_dict[item]['calories'].replace('kcal', '').strip())
                total_carbs += ratio * float(all_food_dict[item]['carbs'].strip())
                total_fat += ratio * float(all_food_dict[item]['fat'].strip())
                total_protein += ratio * float(all_food_dict[item]['protein'].strip())
                total_sugar += ratio * float(all_food_dict[item]['sugar'].strip())
                total_fiber += ratio * float(all_food_dict[item]['fiber'].strip())
        
        # Adding values to return_matrix
        return_dict[dates[i]] = [round(total_calories,1), round(total_carbs,1), round(total_fat,1), round(total_protein,1), round(total_sugar,1), round(total_fiber,1)]
        
    # Returning with matrix
    return return_dict

In [ ]:
# Function for creating int list from pandas column
def pandas_column_to_list(df, column, removable):

    # Getting list from column
    temp_list = list(df[column])
    
    # Replacing removable item with empty string
    list_values = [float(x.replace(removable, '').strip()) for x in temp_list]
    
    # Returning list
    return list_values

In [ ]:
# Function for creating all results matrix
def all_results_matrix(ordered_dates_list, general_info_dict, cardio_list, workout_list, workout_calories_burnt_list, breakfast_cal_macros, lunch_cal_macros, dinner_cal_macros):

    # Creating all results matrix
    all_data_calculated_matrix = []

    # Iterating through all rows
    for i in range(0,len(ordered_dates_list)):

        # Variable for current date
        date = ordered_dates_list[i]

        # Getting general info
        general_info = general_info_dict[date]
        #print(date)

        # Calculating burnt calories and total cardio time
        total_burnt_kcal = 0
        total_cardio_time = 0

        for cardio in cardio_list:
            if general_info[cardio] != '-':
                total_burnt_kcal += int(general_info[cardio]['kcal'])
                total_cardio_time += int(general_info[cardio]['time'].split(' ')[0])

        # Getting calories burnt from real workout
        try:
            total_burnt_kcal += workout_calories_burnt_list[workout_list.index(general_info['workout'])]
            total_cardio_time += 60
        except:
            pass

        # Getting and summing up calories
        # Try/Except is necessary, when not all 3 items are recorded for the day yet
        try:
            breakfast_cals = breakfast_cal_macros[date][0]
        except:
            breakfast_cals = 0
        try:
            lunch_cals = lunch_cal_macros[date][0]
        except:
            lunch_cals = 0
        try:
            dinner_cals = dinner_cal_macros[date][0]
        except:
            dinner_cals = 0
        eaten_calories = np.sum([breakfast_cals, lunch_cals, dinner_cals])

        # Creating column for total calories
        if abs(int(general_info['expected'].split(' ')[0]) - eaten_calories) > 100:
            exceed_emoji = ':x:'
        else:
            exceed_emoji = ':check_mark:'

        # Summing up macros
        all_carbs = breakfast_cal_macros[date][1] + lunch_cal_macros[date][1] + dinner_cal_macros[date][1]
        all_fat = breakfast_cal_macros[date][2] + lunch_cal_macros[date][2] + dinner_cal_macros[date][2]
        all_protein = breakfast_cal_macros[date][3] + lunch_cal_macros[date][3] + dinner_cal_macros[date][3]
        all_sugar = int(round(breakfast_cal_macros[date][4] + lunch_cal_macros[date][4] + dinner_cal_macros[date][4]))
        all_fiber = int(round(breakfast_cal_macros[date][5] + lunch_cal_macros[date][5] + dinner_cal_macros[date][5]))

        # Calculating all calories according to macros
        eaten_calories_acc_macros = all_carbs*4 + all_fat*9 + all_protein*4

        # Calculating percentage of macros intake by calories
        cal_perc_carbs = round(all_carbs*4 / eaten_calories_acc_macros * 100,1)
        cal_perc_fat = round(all_fat*9 / eaten_calories_acc_macros * 100,1)
        cal_perc_prot = round(all_protein*4 / eaten_calories_acc_macros * 100,1)

        # Calculating total kcal balance
        total_kcal_intake = eaten_calories - total_burnt_kcal

        # Creating temp row for different data
        temp_row = [date, general_info['mass'], general_info['workout'], general_info['phase']]
        temp_row += [str(total_cardio_time) + ' min', str(total_burnt_kcal) + ' kcal']
        temp_row += [str(int(total_kcal_intake)) + ' kcal']
        #temp_row += [exceed_emoji]
        temp_row += [str(int(eaten_calories)) + ' kcal', str(int(breakfast_cals)) + ' kcal']
        temp_row += [str(int(lunch_cals)) + ' kcal', str(int(dinner_cals)) + ' kcal']
        temp_row += [str(int(all_carbs)) + 'g', str(int(all_fat)) + 'g']
        temp_row += [str(int(all_protein)) + 'g', str(cal_perc_carbs) + '%'] 
        temp_row += [str(cal_perc_fat) + '%', str(cal_perc_prot) + '%']
        temp_row += [str(all_sugar) + 'g', str(all_fiber) + 'g']

        # Appending to all data matrix
        all_data_calculated_matrix.append(temp_row)
    
    # Returning all results matrix
    return(all_data_calculated_matrix)

In [ ]:
# Function for transforming date to week number of the year
# Returning with string of year + / + week_num
# Example: 2022/31

def date_to_week_num(date):

    # Getting current date
    current_date = date.split('/')

    # Getting week num
    curr_week_num = datetime.date(int(current_date[0]), int(current_date[1]), int(current_date[2])).isocalendar()[1]

    # Creating week num var
    week_num = current_date[0] + '/' + str(curr_week_num)

    # Returning with created string
    return(week_num)

In [ ]:
# Function for extracting date and mass values from dataframe, returning dataframe with avg mass values added
# Parameters:
def df_add_avg_mass_col(df, property, property_name, property_dim, round_val):

    # Getting the weekly avg value of chosen property
    dates_list = list(df['Date'])
    property_list = list(df[property])

    # Creating dictionary for dates and chosen property
    property_dict = {}
    for i in range(0,len(dates_list)):
        property_dict[dates_list[i]] = property_list[i]

    # Creating dictionary of dates with week numbers
    dates_week_nums = {}

    # Filling up dict with appropriate week num keys and empty lists
    for date in dates_list:

        # Creating week number from date
        week_num = date_to_week_num(date)

        # Getting value at the date
        current_val = float(property_dict[date].split(' ')[0])

        # If the week numb already exists
        if week_num in dates_week_nums:
            current_weeknum_property_list = dates_week_nums[week_num]

            # Appending to list
            current_weeknum_property_list.append(current_val)

            # Updating dict value
            dates_week_nums[week_num] = current_weeknum_property_list

        # Else appending to dict
        else:
            dates_week_nums[week_num] = [current_val]


    # Creating new dictionary with the avg value of property
    dates_week_nums_avg = {}
        
    # Creating dictionary with the avg
    for week_num in dates_week_nums:

        # Getting avg of list
        if round_val == 0:
            dates_week_nums_avg[week_num] = round(np.mean(dates_week_nums[week_num]))
        else:  
            dates_week_nums_avg[week_num] = round(np.mean(dates_week_nums[week_num]),round_val)


    # Creating list for avg mass values
    avg_vals = []

    # Creating list to append back to dataframe
    for date in dates_list:

        # Creating week number from date
        week_num = date_to_week_num(date)

        # Appending to avg_vals
        avg_vals.append(str(dates_week_nums_avg[week_num]) + ' ' + property_dim)


    # Appending to dataframe
    df['Avg ' + property_name] = avg_vals

    # Returning with dataframe
    return(df)


In [ ]:
# Function for plotting values
def plotting_series(data, captions_list, ylim, avg_line, marker_colour = 'firebrick', line_colour = 'orange', markersize = 6.4, linewidth = 5):
    
    # Getting captions
    title, label, xlabel, ylabel = captions_list
    
    # Creating figure
    plt.figure(figsize=(10,7))
    
    # If avg_line = True, calculating moving avg
    if avg_line == True:
        
        # List for moving avg
        moving_avg_list = []
        
        # Iterating through datapoints, appending to moving_avg_list and plotting
        for i in range(0,len(data)):
            moving_avg_list.append(np.mean(data[0:i+1]))
            if i == 0:
                plt.plot(i+1, data[i], 'o', color = marker_colour, label = label)
            else:
                plt.plot(i+1, data[i], 'o', color = marker_colour)
            
        # Plotting avg line
        plt.plot(list(range(1,len(moving_avg_list)+1)), moving_avg_list, line_colour, label = 'Moving average', linewidth = linewidth)
        
    # Else only plot the values
    else:
        plt.plot(data, 'o', color = marker_colour, label = label, markersize = markersize)

    # Setting ylim
    plt.ylim(ylim)
    
    # Adding title and labels
    plt.title(title, fontsize=18)
    plt.ylabel(xlabel, fontsize=14)
    plt.xlabel(ylabel, fontsize=14)
    
    # Adding legend, grid and plotting
    plt.legend()
    plt.grid()
    plt.show()

## Getting data

In [ ]:
# Account-specific settings stay in the environment, not in this notebook.
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", ".local/service-account.json")
spreadsheet_id = os.environ.get("MACRO_APP_SPREADSHEET_ID")
if not spreadsheet_id:
    raise ValueError("Set MACRO_APP_SPREADSHEET_ID before running this cell.")

service_account = gspread.service_account(filename=credentials_path)
spreadsheet = service_account.open_by_key(spreadsheet_id)

# Read the daily log, using a configurable worksheet name.
general_matrix, breakfast_matrix, lunch_matrix, dinner_matrix = sheet_to_matrix(
    os.environ.get("MACRO_APP_LOG_WORKSHEET", "daily_log")
)

# Creating dictionaries for meals
breakfast_dict = sheetmatrix_to_dict(breakfast_matrix)
lunch_dict = sheetmatrix_to_dict(lunch_matrix)
dinner_dict = sheetmatrix_to_dict(dinner_matrix)


In [ ]:
# Getting yaml file with exercises
with open(os.environ.get("MACRO_APP_KCAL_CONFIG", "kcal.yaml"), "r") as stream:
    kcal_values = yaml.safe_load(stream)

# Separating into variables
exercise_burnt_kcal_dict = kcal_values['exercise_burnt_kcal_dict']
workout_reps = kcal_values['workout_repetitions']

# Creating list for all workouts and calories burnt per workouts
workouts = ['demo_workout'] 
all_workouts_kcal_burnt = []

# Iterating through all workout dictionaries
for workout_dict in [workout_reps]:

    # Calculating total kcal burnt per workout
    temp_kcal_burnt = 0

    # Iterating through all sets in workout
    for workout_set, workout_rep in workout_dict.items():

        # Getting calories burnt
        temp_kcal_burnt += exercise_burnt_kcal_dict[workout_set] * workout_rep

    # Add the configured warm-up allowance.
    temp_kcal_burnt += kcal_values.get('warmup_kcal', 0)

    # Appending to all calories burnt per workout
    all_workouts_kcal_burnt.append(round(temp_kcal_burnt))

# Creating dictionary of all workouts
workouts_calories_burnt_dict = dict(zip(workouts, all_workouts_kcal_burnt))

In [ ]:
# Activity names and calorie assumptions come from the selected configuration.
cardio_rates = kcal_values['cardio_kcal_per_minute']
cardio_list = list(cardio_rates)
cardio_calories_burnt_list = list(cardio_rates.values())
workout_rates = kcal_values['workout_kcal']
workout_list = list(workout_rates)
workout_calories_burnt_list = list(workout_rates.values())

# Creating header list
general_properties_list = ['mass', 'workout', 'phase', 'expected'] + cardio_list

# Creating dictionary for general info
general_info_dict, ordered_dates_list = general_info_to_dict(general_matrix, general_properties_list, cardio_list, cardio_calories_burnt_list)

In [ ]:
# Getting all food as a matrix
all_food_matrix = sheet_to_matrix(os.environ.get("MACRO_APP_MACROS_WORKSHEET", "macros"), remove_nan = True)

# Getting dictionary of all food items
food_properties_list = ['quantity', 'calories', 'carbs', 'fat', 'protein', 'sugar', 'fiber', 'usual amount', 'type']
all_food_dict, all_food_list = general_info_to_dict(all_food_matrix, food_properties_list)

## Creating dataframe

In [ ]:
# Getting calories, macros breakfast
breakfast_cal_macros = getting_total_macros_cals(breakfast_dict, ordered_dates_list)

# Getting calories, macros lunch
lunch_cal_macros = getting_total_macros_cals(lunch_dict, ordered_dates_list)

# Getting calories, macros dinner
dinner_cal_macros = getting_total_macros_cals(dinner_dict, ordered_dates_list) 

In [ ]:
# Generating all results
all_data_calculated_matrix = all_results_matrix(ordered_dates_list, general_info_dict, cardio_list, workout_list, workout_calories_burnt_list, breakfast_cal_macros, lunch_cal_macros, dinner_cal_macros)

# Creating dataframe 
summary_header = ['Date', 'Mass', 'Day', 'Phase', 'Workout', 
                  'Burnt', 'Σ tot. kcal', 'Σ eaten kcal', 'Breakfast',  
                  'Lunch', 'Dinner', 'Σ carbs', 'Σ fat', 
                  'Σ prot.', '% carbs', '% fat', '% prot.',
                  'Σ sugar', 'Σ fiber']

# Create the pandas DataFrame
summary_df = pd.DataFrame(all_data_calculated_matrix, columns = summary_header)
#summary_df['Done'] = summary_df['Done'].apply(lambda x: emoji.emojize(x, language='alias'))

# Creating avg masses column
summary_df = df_add_avg_mass_col(summary_df, 'Mass', 'mass', 'kg', 1)
summary_df = df_add_avg_mass_col(summary_df, 'Σ eaten kcal', 'eaten kcal', 'kcal', 0)
summary_df = df_add_avg_mass_col(summary_df, 'Σ tot. kcal', 'tot. kcal', 'kcal', 0)

# Rearraning columns
summary_df = summary_df[['Date', 'Mass', 'Avg mass', 'Avg tot. kcal', 'Avg eaten kcal', 'Day', 'Phase', 'Workout', 
                  'Burnt', 'Σ tot. kcal', 'Σ eaten kcal', 'Breakfast',  
                  'Lunch', 'Dinner', 'Σ carbs', 'Σ fat', 
                  'Σ prot.', '% carbs', '% fat', '% prot.',
                  'Σ sugar', 'Σ fiber']]

#summary_df_style = summary_df.style.set_table_attributes('style="font-size: 10.8px"')
summary_df_style = summary_df.style.set_table_attributes('style="font-size: 13px"')

# Checking out dataframe
display(summary_df_style)